# Consolidated Tray Pipeline

This notebook runs the consolidated pipeline:

1. Use YOLO tray segmentation weights to detect the tray and return a perspective-corrected crop.
2. Use the tray classifier checkpoint to predict tray type and confidence.
3. If confidence is below 0.95, fall back to the notebook-3-derived CV grid pipeline for tray dimensions and cell crops.


In [ ]:
from __future__ import annotations

import csv
import json
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import pandas as pd
from ultralytics import YOLO

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.cell_extraction.grid_inference import infer_grid_from_separators
from src.pipeline.run_full_pipeline import run_full_pipeline


In [ ]:
@dataclass
class CFG:
    input_dir: Path = REPO / "data" / "raw"
    output_dir: Path = REPO / "outputs" / "pipeline_notebook"
    yolo_weights: Path = REPO / "models" / "tray_segmentation" / "trayseg_v18_1024.pt"
    tray_checkpoint: Path = REPO / "models" / "tray_classifier" / "best_traytype_net.pth"
    truth_csv: Path | None = None
    tray_type_threshold: float = 0.95
    rectified_width: int = 1400
    save_debug: bool = False
    apply_obliquity_correction: bool = True
    pattern: str = "*.jpg"
    diagnostics_dir: Path = REPO / "outputs" / "pipeline_notebook" / "diagnostics"


cfg = CFG()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
cfg.diagnostics_dir.mkdir(parents=True, exist_ok=True)
cfg


In [ ]:
def load_truth_map(truth_csv: Path | None) -> dict[str, dict[str, int]]:
    if truth_csv is None or not truth_csv.exists():
        return {}

    with truth_csv.open(newline="") as handle:
        reader = csv.DictReader(handle)
        truth_map: dict[str, dict[str, int]] = {}
        for row in reader:
            image_name = Path(row["image"]).name
            truth_map[image_name] = {
                "rows": int(row["rows"]),
                "cols": int(row["cols"]),
            }
        return truth_map


def summarize_results(rows: list[dict]) -> dict[str, float | int | None]:
    summary = {
        "images_processed": len(rows),
        "rectification_failed": sum(1 for row in rows if row["method"] == "rectification_failed"),
        "classifier_route_count": sum(1 for row in rows if row.get("route") == "classifier"),
        "cv_fallback_route_count": sum(1 for row in rows if row.get("route") == "cv_fallback"),
    }
    evaluated = [row for row in rows if "exact_match" in row]
    if evaluated:
        exact = sum(1 for row in evaluated if row["exact_match"])
        rows_ok = sum(1 for row in evaluated if row["rows_match"])
        cols_ok = sum(1 for row in evaluated if row["cols_match"])
        total = len(evaluated)
        summary["images_with_ground_truth"] = total
        summary["exact_match_accuracy"] = exact / total
        summary["rows_accuracy"] = rows_ok / total
        summary["cols_accuracy"] = cols_ok / total
    else:
        summary["images_with_ground_truth"] = 0
        summary["exact_match_accuracy"] = None
        summary["rows_accuracy"] = None
        summary["cols_accuracy"] = None
    return summary


def save_diagnostic_images(warped_bgr, image_stem: str) -> dict[str, str] | None:
    if warped_bgr is None:
        return None

    debug_dir = cfg.diagnostics_dir / image_stem
    debug_dir.mkdir(parents=True, exist_ok=True)
    grid_result = infer_grid_from_separators(
        warped_bgr,
        debug_dir=debug_dir,
        debug_prefix=image_stem,
    )

    longlines_path = debug_dir / f"{image_stem}_sep_longlines.jpg"
    grid_overlay_path = debug_dir / f"{image_stem}_grid_overlay.jpg"
    cv2.imwrite(str(grid_overlay_path), grid_result.overlay_bgr)

    return {
        "longlines": str(longlines_path),
        "grid_overlay": str(grid_overlay_path),
    }


In [ ]:
yolo_model = YOLO(str(cfg.yolo_weights))
truth_map = load_truth_map(cfg.truth_csv)
image_paths = sorted(path for path in cfg.input_dir.glob(cfg.pattern) if path.is_file())
print("Images found:", len(image_paths))
image_paths[:5]


## Single Image Run

In [ ]:
image_path = image_paths[0]

result = run_full_pipeline(
    image=image_path,
    yolo_model=yolo_model,
    tray_type_checkpoint_path=cfg.tray_checkpoint,
    out_dir=cfg.output_dir,
    prefix=image_path.stem,
    tray_type_threshold=cfg.tray_type_threshold,
    rectified_width=cfg.rectified_width,
    save_debug=cfg.save_debug,
    apply_obliquity_correction=cfg.apply_obliquity_correction,
)

diagnostic_paths = save_diagnostic_images(result.warped_bgr, image_path.stem)

single_result = asdict(result)
single_result["warped_bgr"] = None
single_result["diagnostic_paths"] = diagnostic_paths
single_result


In [ ]:
if result.warped_bgr is not None:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(result.warped_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"{Path(result.image_path).name} | method={result.method} | rows={result.rows} cols={result.cols}")
    plt.show()

    if diagnostic_paths is not None:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(cv2.imread(diagnostic_paths["longlines"]), cmap="gray")
        axes[0].axis("off")
        axes[0].set_title("Longlines")
        axes[1].imshow(cv2.cvtColor(cv2.imread(diagnostic_paths["grid_overlay"]), cv2.COLOR_BGR2RGB))
        axes[1].axis("off")
        axes[1].set_title("Grid Overlay")
        plt.show()
else:
    print("No rectified tray image returned.")


## Batch Run

In [ ]:
rows: list[dict] = []
warped_dir = cfg.output_dir / "warped"
warped_dir.mkdir(parents=True, exist_ok=True)

for image_path in image_paths:
    result = run_full_pipeline(
        image=image_path,
        yolo_model=yolo_model,
        tray_type_checkpoint_path=cfg.tray_checkpoint,
        out_dir=cfg.output_dir,
        prefix=image_path.stem,
        tray_type_threshold=cfg.tray_type_threshold,
        rectified_width=cfg.rectified_width,
        save_debug=cfg.save_debug,
        apply_obliquity_correction=cfg.apply_obliquity_correction,
    )

    diagnostic_paths = save_diagnostic_images(result.warped_bgr, image_path.stem)

    if result.warped_bgr is not None:
        cv2.imwrite(str(warped_dir / f"{image_path.stem}.rectified.jpg"), result.warped_bgr)

    row = {
        "image": image_path.name,
        "pred_rows": result.rows,
        "pred_cols": result.cols,
        "tray_type_key": "" if result.tray_type_key is None else "x".join(str(v) for v in result.tray_type_key),
        "tray_type_confidence": result.tray_type_confidence,
        "route": None if result.routing is None else (
            "classifier" if result.routing.use_classifier_layout else "cv_fallback"
        ),
        "method": result.method,
        "reason": result.reason,
        "crop_count": result.crop_count,
        "longlines_path": None if diagnostic_paths is None else diagnostic_paths["longlines"],
        "grid_overlay_path": None if diagnostic_paths is None else diagnostic_paths["grid_overlay"],
    }

    truth = truth_map.get(image_path.name)
    if truth is not None:
        row["true_rows"] = truth["rows"]
        row["true_cols"] = truth["cols"]
        row["rows_match"] = result.rows == truth["rows"]
        row["cols_match"] = result.cols == truth["cols"]
        row["exact_match"] = row["rows_match"] and row["cols_match"]

    rows.append(row)

    payload = asdict(result)
    payload["warped_bgr"] = None
    with (cfg.output_dir / f"{image_path.stem}.result.json").open("w") as handle:
        json.dump(payload, handle, indent=2)

results_df = pd.DataFrame(rows)
results_df.to_csv(cfg.output_dir / "results.csv", index=False)
summary = summarize_results(rows)
with (cfg.output_dir / "summary.json").open("w") as handle:
    json.dump(summary, handle, indent=2)

print("Results written to:", cfg.output_dir)
results_df.head()
